# 🧪 ChurnLens AI
## Week 3 — Statistical Analysis & Hypothesis Testing

### Customer Retention Intelligence & Churn Prediction

**Internship:** YuvaIntern — Virtual Data Science with Python Apprentice Intern  
**Week:** 3 of 5  
**Dataset:** IBM Telco Customer Churn Dataset  

---

### Objective

The objective of Week 3 is to statistically evaluate important customer churn
patterns identified during Week 1 exploratory data analysis and Week 2
advanced visualization.

This week focuses on hypothesis testing, statistical significance, effect size,
and responsible interpretation of results.

### Statistical Tests

1. Chi-Square Test — Contract vs Churn
2. Chi-Square Test — Internet Service vs Churn
3. Chi-Square Test — Payment Method vs Churn
4. Mann–Whitney U Test — Tenure vs Churn
5. Mann–Whitney U Test — Monthly Charges vs Churn

### Significance Level

α = 0.05

In [1]:
import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency
from scipy.stats import mannwhitneyu

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
df = pd.read_csv(
    "../data/processed/Telco-Customer-Churn-Cleaned.csv"
)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (7032, 21)


In [3]:
display(df.head())

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 7032
Columns: 21


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   str    
 1   gender            7032 non-null   str    
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   str    
 4   Dependents        7032 non-null   str    
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   str    
 7   MultipleLines     7032 non-null   str    
 8   InternetService   7032 non-null   str    
 9   OnlineSecurity    7032 non-null   str    
 10  OnlineBackup      7032 non-null   str    
 11  DeviceProtection  7032 non-null   str    
 12  TechSupport       7032 non-null   str    
 13  StreamingTV       7032 non-null   str    
 14  StreamingMovies   7032 non-null   str    
 15  Contract          7032 non-null   str    
 16  PaperlessBilling  7032 non-null   str    
 17  Paymen

In [6]:
missing_values = df.isnull().sum()

print("Missing values:")
display(missing_values[missing_values > 0])

Missing values:


Series([], dtype: int64)

In [7]:
churn_counts = df["Churn"].value_counts()

print("Churn Distribution:")
display(churn_counts)

Churn Distribution:


Churn
No     5163
Yes    1869
Name: count, dtype: int64

In [8]:
churn_percentage = (
    df["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Churn Percentage:")
display(churn_percentage)

Churn Percentage:


Churn
No     73.42
Yes    26.58
Name: proportion, dtype: float64

## 1. Descriptive Statistics

Before performing hypothesis tests, the dataset is examined using descriptive
statistics.

The descriptive analysis provides a baseline understanding of the numerical
variables and helps compare churned and non-churned customers.

In [9]:
numerical_columns = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

display(df[numerical_columns].describe())

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7032.000000,7032.000000,7032.000000,7032.000000
mean,0.162400,32.421786,64.798208,2283.300441
std,0.368844,24.545260,30.085974,2266.771362
min,0.000000,1.000000,18.250000,18.800000
25%,0.000000,9.000000,35.587500,401.450000
50%,0.000000,29.000000,70.350000,1397.475000
75%,0.000000,55.000000,89.862500,3794.737500
max,1.000000,72.000000,118.750000,8684.800000


In [10]:
churn_numeric_summary = (
    df.groupby("Churn")[["tenure", "MonthlyCharges", "TotalCharges"]]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

display(churn_numeric_summary)

tenure                      MonthlyCharges                       \
       count   mean median    std          count   mean median    std   
Churn                                                                   
No      5163  37.65   38.0  24.08           5163  61.31  64.45  31.09   
Yes     1869  17.98   10.0  19.53           1869  74.44  79.65  24.67   

      TotalCharges                             
             count     mean   median      std  
Churn                                          
No            5163  2555.34  1683.60  2329.46  
Yes           1869  1531.80   703.55  1890.82

## 2. Hypothesis Test 1 — Contract Type vs Churn

### Research Question

Is customer churn statistically associated with contract type?

### Null Hypothesis (H₀)

Contract type and customer churn are independent.
There is no statistically significant association between contract type and churn.

### Alternative Hypothesis (H₁)

Contract type and customer churn are not independent.
There is a statistically significant association between contract type and churn.

### Statistical Test

Chi-Square Test of Independence

### Significance Level

α = 0.05

### Decision Rule

- If p-value < 0.05 → Reject H₀
- If p-value ≥ 0.05 → Fail to reject H₀

In [11]:
contract_table = pd.crosstab(
    df["Contract"],
    df["Churn"]
)

print("Contract vs Churn Contingency Table:")
display(contract_table)

Contract vs Churn Contingency Table:


Churn,No,Yes
Contract,,
Month-to-month,2220,1655
One year,1306,166
Two year,1637,48


In [12]:
contract_churn_rate = pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
) * 100

print("Contract-wise Churn Rate (%):")
display(contract_churn_rate.round(2))

Contract-wise Churn Rate (%):


Churn,No,Yes
Contract,,
Month-to-month,57.29,42.71
One year,88.72,11.28
Two year,97.15,2.85


In [13]:
chi2, p_value, degrees_of_freedom, expected = chi2_contingency(
    contract_table
)

print("Chi-Square Statistic:", chi2)
print("p-value:", p_value)
print("Degrees of Freedom:", degrees_of_freedom)

Chi-Square Statistic: 1179.5458287339445
p-value: 7.326182186265472e-257
Degrees of Freedom: 2


In [14]:
expected_table = pd.DataFrame(
    expected,
    index=contract_table.index,
    columns=contract_table.columns
)

print("Expected Frequencies:")
display(expected_table.round(2))

Expected Frequencies:


Churn,No,Yes
Contract,,
Month-to-month,2845.08,1029.92
One year,1080.76,391.24
Two year,1237.15,447.85


In [15]:
print(
    "Minimum expected frequency:",
    expected.min()
)

Minimum expected frequency: 391.23549488054607


In [16]:
if expected.min() >= 5:
    print("Assumption check: Expected frequencies are adequate.")
else:
    print("Warning: Some expected frequencies are below 5.")

Assumption check: Expected frequencies are adequate.


In [17]:
alpha = 0.05

if p_value < alpha:
    decision = "Reject H₀"
    conclusion = (
        "There is a statistically significant association "
        "between contract type and customer churn."
    )
else:
    decision = "Fail to reject H₀"
    conclusion = (
        "There is insufficient statistical evidence to conclude "
        "that contract type and customer churn are associated."
    )

print("Significance Level:", alpha)
print("Decision:", decision)
print("Conclusion:", conclusion)

Significance Level: 0.05
Decision: Reject H₀
Conclusion: There is a statistically significant association between contract type and customer churn.


In [18]:
def cramers_v(table):
    chi2, p, dof, expected = chi2_contingency(table)

    n = table.to_numpy().sum()
    rows, cols = table.shape

    phi2 = chi2 / n

    phi2_corrected = max(
        0,
        phi2 - ((cols - 1) * (rows - 1)) / (n - 1)
    )

    rows_corrected = (
        rows - ((rows - 1) ** 2) / (n - 1)
    )

    cols_corrected = (
        cols - ((cols - 1) ** 2) / (n - 1)
    )

    denominator = min(
        cols_corrected - 1,
        rows_corrected - 1
    )

    return np.sqrt(phi2_corrected / denominator)

In [19]:
cramers_v_value = cramers_v(contract_table)

print("Cramer's V:", round(cramers_v_value, 4))

Cramer's V: 0.4092


In [20]:
contract_result = pd.DataFrame({
    "Test": ["Chi-Square Test"],
    "Variables": ["Contract Type vs Churn"],
    "Chi_Square": [round(chi2, 4)],
    "p_value": [p_value],
    "Degrees_of_Freedom": [degrees_of_freedom],
    "Cramers_V": [round(cramers_v_value, 4)],
    "Significance_Level": [alpha],
    "Decision": [decision]
})

display(contract_result)

,Test,Variables,Chi_Square,p_value,Degrees_of_Freedom,Cramers_V,Significance_Level,Decision
0,Chi-Square Test,Contract Type vs Churn,1179.5458,7.326182e-257,2,0.4092,0.05,Reject H₀


### Interpretation

The Chi-Square Test of Independence was used to evaluate whether contract type
and customer churn are statistically associated.

The test compares the observed customer counts across contract categories with
the counts that would be expected if contract type and churn were independent.

The p-value is compared with the significance level of 0.05.

The result will be interpreted as follows:

- If p-value < 0.05, H₀ will be rejected and the analysis will conclude that
  contract type and churn have a statistically significant association.
- If p-value ≥ 0.05, H₀ will not be rejected because there is insufficient
  statistical evidence of an association.

Cramér's V is additionally reported to describe the strength of the association.

Statistical significance should not be interpreted as proof that contract type
causes customer churn. Other customer characteristics may contribute to the
observed relationship.

## 3. Hypothesis Test 2 — Internet Service vs Churn

### Research Question

Is customer churn statistically associated with Internet Service type?

### Null Hypothesis (H₀)

Internet Service type and customer churn are independent.

### Alternative Hypothesis (H₁)

Internet Service type and customer churn are associated.

### Statistical Test

Chi-Square Test of Independence

### Significance Level

α = 0.05


In [21]:
internet_table = pd.crosstab(
    df["InternetService"],
    df["Churn"]
)

print("Internet Service vs Churn:")
display(internet_table)

Internet Service vs Churn:


Churn,No,Yes
InternetService,,
DSL,1957,459
Fiber optic,1799,1297
No,1407,113


In [22]:
internet_churn_rate = pd.crosstab(
    df["InternetService"],
    df["Churn"],
    normalize="index"
) * 100

print("Internet Service-wise Churn Rate (%):")
display(internet_churn_rate.round(2))

Internet Service-wise Churn Rate (%):


Churn,No,Yes
InternetService,,
DSL,81.00,19.00
Fiber optic,58.11,41.89
No,92.57,7.43


In [23]:
internet_churn_rate = pd.crosstab(
    df["InternetService"],
    df["Churn"],
    normalize="index"
) * 100

print("Internet Service-wise Churn Rate (%):")
display(internet_churn_rate.round(2))

Internet Service-wise Churn Rate (%):


Churn,No,Yes
InternetService,,
DSL,81.00,19.00
Fiber optic,58.11,41.89
No,92.57,7.43


In [24]:
chi2_internet, p_internet, dof_internet, expected_internet = chi2_contingency(
    internet_table
)

print("Chi-Square Statistic:", chi2_internet)
print("p-value:", p_internet)
print("Degrees of Freedom:", dof_internet)

Chi-Square Statistic: 728.6956143058694
p-value: 5.831198962237274e-159
Degrees of Freedom: 2


In [25]:
expected_internet_table = pd.DataFrame(
    expected_internet,
    index=internet_table.index,
    columns=internet_table.columns
)

print("Expected Frequencies:")
display(expected_internet_table.round(2))

print(
    "Minimum expected frequency:",
    expected_internet.min()
)

Expected Frequencies:


Churn,No,Yes
InternetService,,
DSL,1773.86,642.14
Fiber optic,2273.13,822.87
No,1116.01,403.99


Minimum expected frequency: 403.9931740614334


In [26]:
alpha = 0.05

if p_internet < alpha:
    decision_internet = "Reject H₀"
    conclusion_internet = (
        "There is a statistically significant association "
        "between Internet Service and customer churn."
    )
else:
    decision_internet = "Fail to reject H₀"
    conclusion_internet = (
        "There is insufficient statistical evidence to conclude "
        "that Internet Service and customer churn are associated."
    )

print("Significance Level:", alpha)
print("Decision:", decision_internet)
print("Conclusion:", conclusion_internet)

Significance Level: 0.05
Decision: Reject H₀
Conclusion: There is a statistically significant association between Internet Service and customer churn.


In [27]:
cramers_v_internet = cramers_v(internet_table)

print(
    "Cramer's V:",
    round(cramers_v_internet, 4)
)

Cramer's V: 0.3215


In [28]:
internet_result = pd.DataFrame({
    "Test": ["Chi-Square Test"],
    "Variables": ["Internet Service vs Churn"],
    "Chi_Square": [round(chi2_internet, 4)],
    "p_value": [p_internet],
    "Degrees_of_Freedom": [dof_internet],
    "Cramers_V": [round(cramers_v_internet, 4)],
    "Significance_Level": [alpha],
    "Decision": [decision_internet]
})

display(internet_result)

,Test,Variables,Chi_Square,p_value,Degrees_of_Freedom,Cramers_V,Significance_Level,Decision
0,Chi-Square Test,Internet Service vs Churn,728.6956,5.831199e-159,2,0.3215,0.05,Reject H₀


### Interpretation

The Chi-Square Test of Independence was used to determine whether Internet
Service type and customer churn are statistically associated.

The p-value is compared with α = 0.05.

If the p-value is below 0.05, H₀ is rejected and the analysis indicates a
statistically significant association between Internet Service and churn.

If the p-value is 0.05 or greater, H₀ is not rejected because there is
insufficient statistical evidence of an association.

Cramér's V is reported to describe the strength of the association.

The result represents an association in the observed dataset and should not
be interpreted as proof that Internet Service causes customer churn.

## 4. Hypothesis Test 3 — Payment Method vs Churn

### Research Question

Is customer churn statistically associated with Payment Method?

### Null Hypothesis (H₀)

Payment Method and customer churn are independent.

### Alternative Hypothesis (H₁)

Payment Method and customer churn are associated.

### Statistical Test

Chi-Square Test of Independence

### Significance Level

α = 0.05

In [29]:
payment_table = pd.crosstab(
    df["PaymentMethod"],
    df["Churn"]
)

print("Payment Method vs Churn:")
display(payment_table)

Payment Method vs Churn:


Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),1284,258
Credit card (automatic),1289,232
Electronic check,1294,1071
Mailed check,1296,308


In [30]:
payment_churn_rate = pd.crosstab(
    df["PaymentMethod"],
    df["Churn"],
    normalize="index"
) * 100

print("Payment Method-wise Churn Rate (%):")
display(payment_churn_rate.round(2))

Payment Method-wise Churn Rate (%):


Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),83.27,16.73
Credit card (automatic),84.75,15.25
Electronic check,54.71,45.29
Mailed check,80.80,19.20


In [31]:
chi2_payment, p_payment, dof_payment, expected_payment = chi2_contingency(
    payment_table
)

print("Chi-Square Statistic:", chi2_payment)
print("p-value:", p_payment)
print("Degrees of Freedom:", dof_payment)

Chi-Square Statistic: 645.4299001234638
p-value: 1.4263098511063342e-139
Degrees of Freedom: 3


In [32]:
expected_payment_table = pd.DataFrame(
    expected_payment,
    index=payment_table.index,
    columns=payment_table.columns
)

print("Expected Frequencies:")
display(expected_payment_table.round(2))

print(
    "Minimum expected frequency:",
    expected_payment.min()
)

Expected Frequencies:


Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),1132.16,409.84
Credit card (automatic),1116.74,404.26
Electronic check,1736.42,628.58
Mailed check,1177.68,426.32


Minimum expected frequency: 404.2589590443686


In [33]:
alpha = 0.05

if p_payment < alpha:
    decision_payment = "Reject H₀"
    conclusion_payment = (
        "There is a statistically significant association "
        "between Payment Method and customer churn."
    )
else:
    decision_payment = "Fail to reject H₀"
    conclusion_payment = (
        "There is insufficient statistical evidence to conclude "
        "that Payment Method and customer churn are associated."
    )

print("Significance Level:", alpha)
print("Decision:", decision_payment)
print("Conclusion:", conclusion_payment)

Significance Level: 0.05
Decision: Reject H₀
Conclusion: There is a statistically significant association between Payment Method and customer churn.


In [34]:
cramers_v_payment = cramers_v(payment_table)

print(
    "Cramer's V:",
    round(cramers_v_payment, 4)
)

Cramer's V: 0.3023


In [35]:
payment_result = pd.DataFrame({
    "Test": ["Chi-Square Test"],
    "Variables": ["Payment Method vs Churn"],
    "Chi_Square": [round(chi2_payment, 4)],
    "p_value": [p_payment],
    "Degrees_of_Freedom": [dof_payment],
    "Cramers_V": [round(cramers_v_payment, 4)],
    "Significance_Level": [alpha],
    "Decision": [decision_payment]
})

display(payment_result)

,Test,Variables,Chi_Square,p_value,Degrees_of_Freedom,Cramers_V,Significance_Level,Decision
0,Chi-Square Test,Payment Method vs Churn,645.4299,1.426310e-139,3,0.3023,0.05,Reject H₀


### Interpretation

The Chi-Square Test of Independence was used to determine whether Payment
Method and customer churn are statistically associated.

The p-value is compared with α = 0.05.

If the p-value is below 0.05, H₀ is rejected and the analysis indicates a
statistically significant association between Payment Method and churn.

If the p-value is 0.05 or greater, H₀ is not rejected because there is
insufficient statistical evidence of an association.

Cramér's V is reported to describe the strength of the association.

The result represents an association in the observed dataset and should not
be interpreted as proof that Payment Method causes customer churn.

## 5. Hypothesis Test 4 — Tenure vs Churn

### Research Question

Do customers who churn and customers who do not churn have significantly
different tenure distributions?

### Null Hypothesis (H₀)

The tenure distributions of churned and non-churned customers are not
significantly different.

### Alternative Hypothesis (H₁)

The tenure distributions of churned and non-churned customers are
significantly different.

### Statistical Test

Mann–Whitney U Test

### Significance Level

α = 0.05

In [36]:
tenure_no_churn = df.loc[df["Churn"] == "No", "tenure"]
tenure_churn = df.loc[df["Churn"] == "Yes", "tenure"]

print("Non-churned customers:", len(tenure_no_churn))
print("Churned customers:", len(tenure_churn))

Non-churned customers: 5163
Churned customers: 1869


In [37]:
print("Median Tenure - No Churn:", tenure_no_churn.median())
print("Median Tenure - Churn:", tenure_churn.median())

print("Mean Tenure - No Churn:", round(tenure_no_churn.mean(), 2))
print("Mean Tenure - Churn:", round(tenure_churn.mean(), 2))

Median Tenure - No Churn: 38.0
Median Tenure - Churn: 10.0
Mean Tenure - No Churn: 37.65
Mean Tenure - Churn: 17.98


In [38]:
u_statistic_tenure, p_value_tenure = mannwhitneyu(
    tenure_no_churn,
    tenure_churn,
    alternative="two-sided"
)

print("Mann–Whitney U Statistic:", u_statistic_tenure)
print("p-value:", p_value_tenure)

Mann–Whitney U Statistic: 7154668.0
p-value: 6.043046580882717e-211


In [39]:
alpha = 0.05

if p_value_tenure < alpha:
    decision_tenure = "Reject H₀"
    conclusion_tenure = (
        "There is a statistically significant difference "
        "in tenure between churned and non-churned customers."
    )
else:
    decision_tenure = "Fail to reject H₀"
    conclusion_tenure = (
        "There is insufficient statistical evidence to conclude "
        "that tenure differs significantly between the two groups."
    )

print("Significance Level:", alpha)
print("Decision:", decision_tenure)
print("Conclusion:", conclusion_tenure)

Significance Level: 0.05
Decision: Reject H₀
Conclusion: There is a statistically significant difference in tenure between churned and non-churned customers.


### Interpretation

The Mann–Whitney U Test was used to compare the tenure distributions of
churned and non-churned customers.

The non-churned group contained 5,163 customers, while the churned group
contained 1,869 customers.

The median tenure was 38 months for non-churned customers and 10 months
for churned customers. The mean tenure was 37.65 months for non-churned
customers and 17.98 months for churned customers.

The Mann–Whitney U statistic was 7,154,668.0 and the p-value was
approximately 6.04 × 10⁻²¹¹.

Since the p-value is much smaller than α = 0.05, the null hypothesis (H₀)
is rejected.

Therefore, there is statistically significant evidence that tenure
distributions differ between churned and non-churned customers.

The result indicates a strong difference in customer tenure between the
two groups in this dataset. However, statistical significance does not
establish that tenure alone causes customer churn.

## 6. Hypothesis Test 5 — Monthly Charges vs Churn

### Research Question

Do monthly charges differ significantly between churned and non-churned
customers?

### Null Hypothesis (H₀)

The monthly charge distributions of churned and non-churned customers
are not significantly different.

### Alternative Hypothesis (H₁)

The monthly charge distributions of churned and non-churned customers
are significantly different.

### Statistical Test

Mann–Whitney U Test

### Significance Level

α = 0.05

In [40]:
monthly_no_churn = df.loc[
    df["Churn"] == "No",
    "MonthlyCharges"
]

monthly_churn = df.loc[
    df["Churn"] == "Yes",
    "MonthlyCharges"
]

print("Non-churned customers:", len(monthly_no_churn))
print("Churned customers:", len(monthly_churn))

Non-churned customers: 5163
Churned customers: 1869


In [41]:
print(
    "Median Monthly Charges - No Churn:",
    monthly_no_churn.median()
)

print(
    "Median Monthly Charges - Churn:",
    monthly_churn.median()
)

print(
    "Mean Monthly Charges - No Churn:",
    round(monthly_no_churn.mean(), 2)
)

print(
    "Mean Monthly Charges - Churn:",
    round(monthly_churn.mean(), 2)
)

Median Monthly Charges - No Churn: 64.45
Median Monthly Charges - Churn: 79.65
Mean Monthly Charges - No Churn: 61.31
Mean Monthly Charges - Churn: 74.44


In [42]:
u_statistic_monthly, p_value_monthly = mannwhitneyu(
    monthly_no_churn,
    monthly_churn,
    alternative="two-sided"
)

print(
    "Mann–Whitney U Statistic:",
    u_statistic_monthly
)

print(
    "p-value:",
    p_value_monthly
)

Mann–Whitney U Statistic: 3663498.5
p-value: 8.467195044548749e-54


In [43]:
alpha = 0.05

if p_value_monthly < alpha:
    decision_monthly = "Reject H₀"
    conclusion_monthly = (
        "There is a statistically significant difference "
        "in monthly charges between churned and non-churned customers."
    )
else:
    decision_monthly = "Fail to reject H₀"
    conclusion_monthly = (
        "There is insufficient statistical evidence to conclude "
        "that monthly charges differ significantly between the two groups."
    )

print("Significance Level:", alpha)
print("Decision:", decision_monthly)
print("Conclusion:", conclusion_monthly)

Significance Level: 0.05
Decision: Reject H₀
Conclusion: There is a statistically significant difference in monthly charges between churned and non-churned customers.


### Interpretation

The Mann–Whitney U Test was used to compare the monthly charge
distributions of churned and non-churned customers.

The non-churned group contained 5,163 customers, while the churned group
contained 1,869 customers.

The median monthly charge was 64.45 for non-churned customers and 79.65
for churned customers. The mean monthly charge was 61.31 for non-churned
customers and 74.44 for churned customers.

The Mann–Whitney U statistic was 3,663,498.5 and the p-value was
approximately 8.47 × 10⁻⁵⁴.

Since the p-value is much smaller than α = 0.05, the null hypothesis (H₀)
is rejected.

Therefore, there is statistically significant evidence that monthly charge
distributions differ between churned and non-churned customers.

The results indicate that churned customers tend to have higher monthly
charges in the observed dataset. However, statistical significance does
not establish that higher monthly charges alone cause customer churn.

## 7. Effect Size Analysis

Statistical significance indicates whether the observed result is unlikely
to have occurred by chance under the null hypothesis.

Effect size provides additional information about the practical magnitude
or strength of the observed relationship.

### Effect Size Measures

- Cramér's V → Chi-Square tests
- Rank-Biserial Correlation → Mann–Whitney U tests

Effect sizes should be interpreted together with p-values and the underlying
business context.

In [44]:
def rank_biserial(u_statistic, n1, n2):
    return 1 - (2 * u_statistic) / (n1 * n2)

In [45]:
n1 = len(tenure_no_churn)
n2 = len(tenure_churn)

rank_biserial_tenure = rank_biserial(
    u_statistic_tenure,
    n1,
    n2
)

print(
    "Rank-Biserial Correlation - Tenure:",
    round(rank_biserial_tenure, 4)
)

Rank-Biserial Correlation - Tenure: -0.4829


In [46]:
n1_monthly = len(monthly_no_churn)
n2_monthly = len(monthly_churn)

rank_biserial_monthly = rank_biserial(
    u_statistic_monthly,
    n1_monthly,
    n2_monthly
)

print(
    "Rank-Biserial Correlation - Monthly Charges:",
    round(rank_biserial_monthly, 4)
)

Rank-Biserial Correlation - Monthly Charges: 0.2407


## 8. Complete Statistical Test Results

In [47]:
final_results = pd.DataFrame({
    "Test": [
        "Chi-Square",
        "Chi-Square",
        "Chi-Square",
        "Mann–Whitney U",
        "Mann–Whitney U"
    ],
    "Variables": [
        "Contract vs Churn",
        "Internet Service vs Churn",
        "Payment Method vs Churn",
        "Tenure vs Churn",
        "Monthly Charges vs Churn"
    ],
    "Test_Statistic": [
        chi2,
        chi2_internet,
        chi2_payment,
        u_statistic_tenure,
        u_statistic_monthly
    ],
    "p_value": [
        p_value,
        p_internet,
        p_payment,
        p_value_tenure,
        p_value_monthly
    ],
    "Effect_Size": [
        cramers_v_value,
        cramers_v_internet,
        cramers_v_payment,
        rank_biserial_tenure,
        rank_biserial_monthly
    ],
    "Effect_Size_Type": [
        "Cramer's V",
        "Cramer's V",
        "Cramer's V",
        "Rank-Biserial",
        "Rank-Biserial"
    ],
    "Significance_Level": [
        0.05,
        0.05,
        0.05,
        0.05,
        0.05
    ],
    "Decision": [
        decision,
        decision_internet,
        decision_payment,
        decision_tenure,
        decision_monthly
    ]
})

final_results["Test_Statistic"] = final_results["Test_Statistic"].round(4)
final_results["p_value"] = final_results["p_value"].apply(
    lambda x: f"{x:.4e}"
)
final_results["Effect_Size"] = final_results["Effect_Size"].round(4)

display(final_results)

,Test,Variables,Test_Statistic,p_value,Effect_Size,Effect_Size_Type,Significance_Level,Decision
0,Chi-Square,Contract vs Churn,1.179546e+03,7.3262e-257,0.4092,Cramer's V,0.05,Reject H₀
1,Chi-Square,Internet Service vs Churn,7.286956e+02,5.8312e-159,0.3215,Cramer's V,0.05,Reject H₀
2,Chi-Square,Payment Method vs Churn,6.454299e+02,1.4263e-139,0.3023,Cramer's V,0.05,Reject H₀
3,Mann–Whitney U,Tenure vs Churn,7.154668e+06,6.0430e-211,-0.4829,Rank-Biserial,0.05,Reject H₀
4,Mann–Whitney U,Monthly Charges vs Churn,3.663498e+06,8.4672e-54,0.2407,Rank-Biserial,0.05,Reject H₀


# 9. Statistical Findings & Business Interpretation

The statistical analysis was conducted to validate important churn patterns
identified during the exploratory and visualization stages of the project.

A significance level of α = 0.05 was used for all hypothesis tests.

The analysis included Chi-Square Tests of Independence for categorical
variables and Mann–Whitney U Tests for numerical variables.

## 9.1 Overall Findings

The statistical analysis provides evidence that several customer attributes
are significantly associated with churn or differ significantly between
churned and non-churned customers.

The analysis examined:

1. Contract Type vs Churn
2. Internet Service vs Churn
3. Payment Method vs Churn
4. Tenure vs Churn
5. Monthly Charges vs Churn

The hypothesis tests provide statistical support for the patterns observed
during Week 1 Exploratory Data Analysis and Week 2 Advanced Visualization.

However, statistical significance should not be interpreted as proof of
causation. The results identify relationships and differences within the
observed dataset.

## 9.2 Finding 1 — Contract Type and Churn

The Chi-Square Test of Independence was used to examine the relationship
between contract type and customer churn.

The test evaluates whether contract type and churn are statistically
independent.

The analysis found a statistically significant association between
Contract Type and Churn because the obtained p-value was below the
significance level of α = 0.05.

The Week 1 and Week 2 analysis also showed substantial differences in
churn rates across contract categories, with month-to-month customers
showing a considerably higher churn rate than customers with longer-term
contracts.

### Business Interpretation

Contract structure can therefore be considered an important customer
retention dimension.

From a business perspective, customers on month-to-month contracts may
require greater retention attention. The company could consider targeted
retention campaigns, loyalty benefits, contract upgrade incentives, or
personalized offers for customers approaching renewal or showing other
risk indicators.

However, these findings represent statistical association and do not prove
that contract type alone causes churn.

## 9.3 Finding 2 — Internet Service and Churn

The Chi-Square Test of Independence was used to determine whether Internet
Service and customer churn were statistically associated.

The analysis indicated a statistically significant association between
Internet Service and Churn because the p-value was below α = 0.05.

This confirms that the differences observed during the exploratory and
visualization stages are statistically meaningful in the analyzed dataset.

### Business Interpretation

Internet service type can be considered an important segmentation variable
for customer retention analysis.

The business could investigate whether particular service categories are
associated with customer dissatisfaction, pricing concerns, service
quality issues, or differences in customer expectations.

Targeted service improvement and customer-support initiatives could then
be designed for higher-churn segments.

The result should be interpreted as an association rather than evidence
that Internet Service directly causes churn.

## 9.4 Finding 3 — Payment Method and Churn

The Chi-Square Test of Independence produced a Chi-Square statistic of
645.4299 with 3 degrees of freedom.

The p-value was approximately 1.43 × 10⁻¹³⁹, which is substantially below
the significance level of α = 0.05.

Therefore, the null hypothesis was rejected.

The Cramér's V value was 0.3023, indicating a moderate association between
Payment Method and customer churn.

### Business Interpretation

Payment Method is statistically associated with customer churn in the
observed dataset.

This suggests that payment-related customer segments deserve attention
during retention analysis.

A business could investigate whether differences in payment methods are
related to payment convenience, customer experience, billing friction,
automatic-payment adoption, or other customer characteristics.

Possible actions include improving payment convenience, communicating
billing information more clearly, encouraging convenient payment options,
and monitoring customers who experience payment-related difficulties.

However, the statistical result does not prove that payment method itself
causes customer churn.

## 9.5 Finding 4 — Tenure and Churn

The Mann–Whitney U Test was used to compare tenure distributions between
churned and non-churned customers.

There were 5,163 non-churned customers and 1,869 churned customers.

The median tenure was:

- Non-churned customers: 38 months
- Churned customers: 10 months

The mean tenure was:

- Non-churned customers: 37.65 months
- Churned customers: 17.98 months

The Mann–Whitney U statistic was 7,154,668.0 and the p-value was
approximately 6.04 × 10⁻²¹¹.

Since the p-value is substantially below α = 0.05, the null hypothesis
was rejected.

The rank-biserial correlation was -0.4829. The negative direction reflects
the ordering of the two groups used in the calculation, while the magnitude
indicates a substantial difference between the groups.

### Business Interpretation

Customer tenure is one of the most important findings of the statistical
analysis.

Customers who churned had substantially shorter tenure than customers who
remained with the service.

This suggests that the early customer lifecycle may deserve particular
retention attention.

A business could develop an early-stage customer retention program,
including onboarding support, satisfaction checks, personalized
communication, service guidance, and targeted offers during the first
months of the customer relationship.

The result identifies a strong difference between the two groups but does
not establish that shorter tenure itself causes churn.

## 9.6 Finding 5 — Monthly Charges and Churn

The Mann–Whitney U Test was used to compare monthly charge distributions
between churned and non-churned customers.

There were 5,163 non-churned customers and 1,869 churned customers.

The median monthly charges were:

- Non-churned customers: 64.45
- Churned customers: 79.65

The mean monthly charges were:

- Non-churned customers: 61.31
- Churned customers: 74.44

The Mann–Whitney U statistic was 3,663,498.5 and the p-value was
approximately 8.47 × 10⁻⁵⁴.

Since the p-value is substantially below α = 0.05, the null hypothesis
was rejected.

The rank-biserial correlation was 0.2407, indicating a smaller but
meaningful difference between the two groups.

### Business Interpretation

Churned customers had higher monthly charges than non-churned customers
in the observed dataset.

This suggests that pricing and perceived value may be useful dimensions
for customer retention analysis.

The business could investigate whether customers with higher monthly
charges receive sufficient perceived value from their services.

Potential actions include personalized plans, value-based offers,
transparent billing communication, loyalty benefits, and targeted
retention campaigns for appropriate customer segments.

The result represents a statistical difference between groups and does
not prove that higher monthly charges directly cause churn.

## 9.7 Executive Business Summary

The Week 3 statistical analysis confirms that important customer
characteristics are significantly related to churn patterns in the
analyzed dataset.

The major findings are:

| Area | Statistical Finding | Business Implication |
|---|---|---|
| Contract Type | Significant association with churn | Focus retention efforts on higher-risk contract segments |
| Internet Service | Significant association with churn | Investigate service-specific customer experience |
| Payment Method | Significant association; Cramér's V = 0.3023 | Review payment experience and billing convenience |
| Tenure | Significant difference; Rank-Biserial = -0.4829 | Strengthen early customer lifecycle engagement |
| Monthly Charges | Significant difference; Rank-Biserial = 0.2407 | Investigate pricing and perceived customer value |

Overall, the results suggest that customer retention strategies should
consider multiple dimensions rather than relying on a single variable.

The statistical findings can be used as a foundation for the machine
learning stage of the project, where multiple customer attributes will be
combined to develop predictive churn models.

## 9.8 Statistical Interpretation Limitations

The statistical tests identify significant associations or differences
within the analyzed dataset. They do not establish causal relationships.

A statistically significant result does not necessarily mean that a
variable independently causes churn.

Additionally, the dataset represents a specific customer population and
business context. Therefore, findings should be validated against current
business data before being used for operational decision-making.

The analysis also focuses on individual variables. Customer churn is a
complex outcome that may depend on interactions among multiple customer,
service, pricing, and behavioral characteristics.

These limitations motivate the next stage of the project: Machine Learning
Model Development and Evaluation.